# Lab: Text Classification with TorchText

## 1. Introduction

Text classification is a fundamental Task in Natural Language Processing (NLP). 
The goal is to assign a Label (e.g., "Sports", "Politics") to a given Text.

In this lab, we will:
1.  Load the **AG_NEWS** dataset (News Classification).
2.  Build a **Data Pipeline** using `torchtext`.
3.  Understand **EmbeddingBag** for efficiency.
4.  Train a robust **Text Classifier**.

### The Dataset: AG_NEWS
*   **Class 1**: World
*   **Class 2**: Sports
*   **Class 3**: Business
*   **Class 4**: Sci/Tech



In [1]:
import torch
from torchtext.datasets import AG_NEWS

# 1. Inspect the Data
train_iter = iter(AG_NEWS(split='train'))

print("--- Sample Data ---")
for i in range(3):
    label, text = next(train_iter)
    print(f"Label: {label} | Text: {text}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Python312\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:\Python312\Lib\site-packages\tornado\platform\asyncio.py", line 211, in 

ModuleNotFoundError: Package `torchdata` not found. Please install following instructions at https://github.com/pytorch/data

## 2. Preprocessing Pipeline

To feed text into a Neural Network, we must convert it into numbers (Indices).

1.  **Tokenizer**: Splits sentences into words.
2.  **Vocabulary**: Maps every unique word to an Integer ID.



In [2]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

tokenizer = get_tokenizer('basic_english')
train_iter = AG_NEWS(split='train')

def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

# Build Vocabulary
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

print(f"Vocabulary Size: {len(vocab)}")
print(f"Index for 'computer': {vocab['computer']}")


ModuleNotFoundError: Package `torchdata` not found. Please install following instructions at https://github.com/pytorch/data

## 3. Data Loading and Collation

### Text & Label Pipelines
We create simple helper functions to transform raw data:
*   `text_pipeline`: String -> List of Integers
*   `label_pipeline`: Label (1-4) -> Index (0-3)

### The Collate Function (Critical Concept)
Since sentences have **different lengths**, we cannot just stack them into a square matrix straight away.
For `EmbeddingBag`, we need a flat list of text and a list of **offsets** (indices where each new sentence starts).

*   **text_list**: All words from a batch flattened into 1D tensor.
*   **offsets**: [0, len(sent1), len(sent1)+len(sent2), ...]



In [3]:
text_pipeline = lambda x: vocab(tokenizer(x))
label_pipeline = lambda x: int(x) - 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def collate_batch(batch):
    label_list, text_list, offsets = [], [], [0]
    
    for (_label, _text) in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        offsets.append(processed_text.size(0))
        
    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    
    return label_list.to(device), text_list.to(device), offsets.to(device)

# Test the function
from torch.utils.data import DataLoader
train_iter = AG_NEWS(split='train')
dataloader = DataLoader(train_iter, batch_size=8, shuffle=False, collate_fn=collate_batch)

# Inspect one batch
labels, text, offsets = next(iter(dataloader))
print(f"Labels: {labels}")
print(f"Offsets: {offsets}")
print(f"Text Stream Shape: {text.shape}")


ModuleNotFoundError: Package `torchdata` not found. Please install following instructions at https://github.com/pytorch/data

## 4. The Model: FastText Architecture

We use `nn.EmbeddingBag`. 
Unlike `nn.Embedding` (which returns a matrix [Seq, Dim]), `nn.EmbeddingBag` computes the **mean** (average) of embeddings for the sequence.

**Why?**
*   It is incredibly **computationally efficient**.
*   It handles variable length sequences automatically (using offsets).
*   For classification tasks, the "average meaning" of words is often sufficient.



In [ ]:
import torch.nn as nn

class TextClassificationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassificationModel, self).__init__()
        
        # EmbeddingBag: Replaces Embedding + Mean/Sum Pooling
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=True)
        
        # Classifier
        self.fc = nn.Linear(embed_dim, num_class)
        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

# Hyperparameters
num_class = len(set([label for (label, text) in AG_NEWS(split='train')]))
vocab_size = len(vocab)
EMBED_DIM = 64
BATCH_SIZE = 64

model = TextClassificationModel(vocab_size, EMBED_DIM, num_class).to(device)
print(model)


## 5. Training Loop

We split the training data into **Train** (95%) and **Validation** (5%) sets.



In [ ]:
import time
from torch.utils.data.dataset import random_split
from torchtext.data.functional import to_map_style_dataset

# Prepare Datasets
train_iter, test_iter = AG_NEWS()
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

num_train = int(len(train_dataset) * 0.95)
split_train_, split_valid_ = random_split(train_dataset, [num_train, len(train_dataset) - num_train])

train_dataloader = DataLoader(split_train_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(split_valid_, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)

# Training Helpers
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=4.0)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.9)

def train(dataloader):
    model.train()
    total_acc, total_count = 0, 0
    start_time = time.time()

    for idx, (label, text, offsets) in enumerate(dataloader):
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)

    return total_acc / total_count

def evaluate(dataloader):
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        for idx, (label, text, offsets) in enumerate(dataloader):
            predicted_label = model(text, offsets)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc / total_count

# --- Run Training ---
EPOCHS = 5 # Small number for demo
print("Starting Training...")

for epoch in range(1, EPOCHS + 1):
    acc_train = train(train_dataloader)
    acc_val = evaluate(valid_dataloader)
    scheduler.step()
    print(f"| Epoch {epoch} | Train Acc: {acc_train:.3f} | Valid Acc: {acc_val:.3f} |")


## 6. Real-world Inference

Let's test the model on custom headlines.



In [ ]:
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

def predict(text, text_pipeline):
    with torch.no_grad():
        text = torch.tensor(text_pipeline(text))
        output = model(text, torch.tensor([0]))
        return output.argmax(1).item() + 1

# Examples
str1 = "Manchester United won the match against Arsenal"
str2 = "Apple announces new iPhone with AI features"
str3 = "Stock markets crash globally due to recession fears"

print(f"'{str1}' -> {ag_news_label[predict(str1, text_pipeline)]}")
print(f"'{str2}' -> {ag_news_label[predict(str2, text_pipeline)]}")
print(f"'{str3}' -> {ag_news_label[predict(str3, text_pipeline)]}")


## Conclusion

Classification of texts is one of the major tasks in Natural Language Processing (NLP).

RNNs are a powerful tool for this task, but they can be slow and resource-intensive.

Try to optimize the code for better performance.

Consider using a GPU for training and adjust the batch size accordingly.

Hyperparameter tuning can also be used to improve the performance of the model.

